# 🧪 Notebook 03 — Advanced Spatio-Temporal Preprocessing
## Air Quality Analysis — Karachi, Pakistan

This notebook implements a publication-ready preprocessing pipeline designed to handle the nuances of satellite-derived air quality data.

### 🛠️ Key Pipeline Features:
1. **ERA5 Meteorological Refinement:** Unit conversion (K → °C, Pa → hPa) and artifact removal.
2. **3-Stage Smart Imputation:** 
   - *Short-term (≤7 days):* Linear interpolation (captures local trends).
   - *Medium-term:* Seasonal station-wise mean filling.
   - *Long-term:* KNN Imputation using spatial and meteorological features.
3. **Resilient Outlier Handling:** 3×IQR Winsorizing to preserve extreme but real pollution events (e.g., SITE industrial spikes).
4. **Original Feature Engineering:** The **Stagnation Index** (DTR / Wind Speed) to quantify atmospheric trapping.
5. **Multi-Model Scalers:** Preparation of raw, StandardScaled, and MinMaxScaled versions for Tree-based, SVR, and LSTM models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from scipy.stats.mstats import winsorize
from pathlib import Path
import os

# ── Configuration ────────────────────────────────────────────────────────────
RAW_DIR = Path('data/raw')
PROCESSED_DIR = Path('data/processed')
os.makedirs(PROCESSED_DIR, exist_ok=True)

MET_FILE = RAW_DIR / 'karachi_era5_meteo_2019_2024.csv'
POLLUTANT_FILES = list(RAW_DIR.glob('karachi_s5p_*_v2.csv'))

print(f"✓ Processed directory ready: {PROCESSED_DIR}")

## 1. Load and Refine Meteorology (ERA5)

In [ ]:
if not MET_FILE.exists():
    raise FileNotFoundError(f"ERA5 file not found at {MET_FILE}")

met = pd.read_csv(MET_FILE)
met['date'] = pd.to_datetime(met['date'])

# ── Unit Conversions ──
# Kelvin to Celsius
for col in ['temperature_2m', 'temperature_2m_max', 'temperature_2m_min', 'dewpoint_temperature_2m']:
    met[col] = met[col] - 273.15

# Pascal to hPa
met['surface_pressure'] = met['surface_pressure'] / 100.0

# ── Clean Up ──
drop_cols = ['.geo', 'system:index']
met = met.drop(columns=[c for c in drop_cols if c in met.columns])

# ── Engineer Stagnation Index ──
met['dtr'] = met['temperature_2m_max'] - met['temperature_2m_min']
met['stagnation_index'] = met['dtr'] / (met['wind_speed_10m'] + 0.1)

print(f"✓ ERA5 refined: {met.shape}")
met.head(2)

## 2. Load and Merge S5P Pollutants

In [ ]:
pollutant_dfs = []
band_rename = {
    'absorbing_aerosol_index': 'aer_ai',
    'NO2_column_number_density': 'no2',
    'SO2_column_number_density': 'so2',
    'CO_column_number_density': 'co'
}

for f in POLLUTANT_FILES:
    temp_df = pd.read_csv(f)
    temp_df['date'] = pd.to_datetime(temp_df['date'])
    
    # Identify the value column
    val_col = [c for c in temp_df.columns if c in band_rename.keys()]
    if val_col:
        temp_df = temp_df.rename(columns={val_col[0]: band_rename[val_col[0]]})
        pollutant_dfs.append(temp_df[['date', 'station', band_rename[val_col[0]]]])

if pollutant_dfs:
    from functools import reduce
    df_pollutants = reduce(lambda left, right: pd.merge(left, right, on=['date', 'station'], how='outer'), pollutant_dfs)
    # Join with met (broadcast met across all stations)
    df = pd.merge(df_pollutants, met, on='date', how='left')
else:
    print("⚠️ No pollutant files found. Creating shell from met + stations.")
    stations = ['Gulshan-e-Iqbal', 'Saddar', 'SITE_Industrial', 'Korangi_Industrial', 
                'North_Nazimabad', 'Gulistan_Jauhar', 'Landhi', 'Federal_B_Area']
    df_shell = pd.concat([met.assign(station=s) for s in stations])
    # Add empty pollutant columns
    for p in band_rename.values():
        df_shell[p] = np.nan
    df = df_shell

df = df.sort_values(['station', 'date']).reset_index(drop=True)
print(f"✓ Master merge complete: {df.shape}")

## 3. 3-Stage Smart Imputation

In [ ]:
pollutants = ['aer_ai', 'no2', 'so2', 'co']

def smart_impute(group):
    # Stage 1: Linear interpolation for short gaps (<= 7 days)
    group[pollutants] = group[pollutants].interpolate(method='linear', limit=7)
    
    # Stage 2: Fill with Monthly Station Mean
    group['month'] = group['date'].dt.month
    for p in pollutants:
        group[p] = group[p].fillna(group.groupby('month')[p].transform('mean'))
    return group

df = df.groupby('station', group_keys=False).apply(smart_impute)

# Stage 3: KNN Imputation for persistent gaps (using met features)
impute_cols = pollutants + ['temperature_2m', 'wind_speed_10m', 'relative_humidity', 'stagnation_index']
knn = KNNImputer(n_neighbors=5)
df[impute_cols] = knn.fit_transform(df[impute_cols])

print("✓ Imputation pipeline complete. Nulls remaining:")
print(df[pollutants].isnull().sum())

## 4. Winsorizing & Scaling

In [ ]:
# ── 3×IQR Winsorizing ──
def winsorize_3iqr(x):
    q1 = x.quantile(0.25)
    q3 = x.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 3 * iqr
    upper = q3 + 3 * iqr
    return x.clip(lower, upper)

for p in pollutants:
    df[p] = winsorize_3iqr(df[p])

# ── Multi-Scaling ──
exclude = ['date', 'station', 'month']
feature_cols = [c for c in df.columns if c not in exclude]

# 1. Standard Scaler (for SVR/Regression)
std_scaler = StandardScaler()
df_std = df.copy()
df_std[feature_cols] = std_scaler.fit_transform(df[feature_cols])

# 2. MinMax Scaler (for LSTM)
mm_scaler = MinMaxScaler()
df_mm = df.copy()
df_mm[feature_cols] = mm_scaler.fit_transform(df[feature_cols])

# Save variations
df.to_csv(PROCESSED_DIR / 'karachi_final_raw.csv', index=False)
df_std.to_csv(PROCESSED_DIR / 'karachi_final_std.csv', index=False)
df_mm.to_csv(PROCESSED_DIR / 'karachi_final_mm.csv', index=False)

print(f"✅ Preprocessing Complete! Files saved in {PROCESSED_DIR}")